# 4. Analysis

*AI assistance (Claude-sonnet-4.6 in UvA AI Chat): notebook structure and modelling pipeline.*

This notebook produces all main results for RQ1 and RQ2. It runs four steps:

| Step | Input | Output |
|------|-------|--------|
| 4.1 | `analysis_dataset.csv` | Descriptive statistics |
| 4.2 | `analysis_dataset.csv` | RQ1 logistic regression models |
| 4.3 | `politeness_profiles.csv` + `analysis_dataset.csv` | RQ2 politeness profiles |
| 4.4 | All models | Robustness checks |

**RQ1 (replication):** To what extent do different bot roles, compared to human
editors, affect newcomer retention, and have these effects changed since Zheng et al. (2019)?

**RQ2 (extension):** To what extent do politeness strategies directed at newcomers
differ across bot roles and human editors, and do these differences explain
variation in newcomer retention?

### 4.0 Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import warnings
from pathlib import Path

# ── Third-party ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# ── Working directory ────────────────────────────────────────────────────────
os.chdir("/Users/ada/bdacaII-2026-ada")

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42

# ── Data directories ──────────────────────────────────────────────────────────
DATA_DIR      = Path("Final_project/data")
PROCESSED_DIR = DATA_DIR / "processed"
FIGURES_DIR   = DATA_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Input paths ───────────────────────────────────────────────────────────────
ANALYSIS_CSV          = PROCESSED_DIR / "analysis_dataset.csv"
POLITENESS_PROFILES_CSV = PROCESSED_DIR / "politeness_profiles.csv"

# ── Model settings ────────────────────────────────────────────────────────────
# Contact dummies for role-based analysis — excludes unidentified bots
# (excluded from role-based analyses per dossier bias mitigation strategy)
# and human (reference category in logistic regression).
BOT_CONTACT_COLS = [
    "contact_fixer",
    "contact_tagger",
    "contact_notifier",
    "contact_clerk",
    "contact_advisor",
]

# All predictors for the full model (RQ1 Model 2)
ALL_CONTACT_COLS = ["contact_human"] + BOT_CONTACT_COLS

# Cross-validation settings
N_CV_FOLDS = 5   # stratified k-fold folds

print(f"pandas      {pd.__version__}")
print(f"numpy       {np.__version__}")
print(f"Figures dir : {FIGURES_DIR}")

### 4.1 Load data and descriptive statistics

Load the analysis dataset and produce descriptive statistics for all key
variables before modelling.

In [ ]:
def load_analysis_data(path: Path = ANALYSIS_CSV) -> pd.DataFrame:
    """Load the analysis dataset produced by notebook 02.

    Parameters
    ----------
    path : Path to analysis_dataset.csv.

    Returns
    -------
    pd.DataFrame with one row per newcomer.

    Raises
    ------
    FileNotFoundError
        If analysis_dataset.csv does not exist. Run notebook 02 first.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Analysis dataset not found at {path}.\n"
            "Run notebook 02 first to generate it."
        )
    return pd.read_csv(path)


df = load_analysis_data()

print(f"Sample size : {len(df):,}")
print(f"Retention   : {df['retained'].sum():,} ({df['retained'].mean():.1%})")
print(f"\n{'Variable':35s} {'Mean':>8} {'SD':>8} {'Min':>6} {'Max':>6}")
print("-" * 65)

desc_cols = ["session_edits", "retained"] + ALL_CONTACT_COLS
for col in desc_cols:
    print(
        f"{col:35s} "
        f"{df[col].mean():8.3f} "
        f"{df[col].std():8.3f} "
        f"{df[col].min():6.0f} "
        f"{df[col].max():6.0f}"
    )

In [ ]:
# ── Figure 1: Contact rates by sender type ───────────────────────────────────
contact_cols = ALL_CONTACT_COLS + ["contact_unidentified_bot"]
contact_rates = df[contact_cols].mean().sort_values(ascending=False)

# Clean labels for the figure
label_map = {
    "contact_human":            "Human",
    "contact_fixer":            "Fixer",
    "contact_tagger":           "Tagger",
    "contact_notifier":         "Notifier",
    "contact_clerk":            "Clerk",
    "contact_advisor":          "Advisor",
    "contact_unidentified_bot": "Unidentified bot\n(excluded from analysis)",
}
labels = [label_map.get(c, c) for c in contact_rates.index]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(labels, contact_rates.values, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Proportion of newcomers contacted")
ax.set_title(
    "Figure 1. Proportion of newcomers contacted by each sender type\n"
    "within 60 days of registration (N = 5,845)"
)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.invert_yaxis()

# Annotate bars with percentages
for bar, val in zip(bars, contact_rates.values):
    ax.text(
        val + 0.003, bar.get_y() + bar.get_height() / 2,
        f"{val:.1%}", va="center", fontsize=9
    )

plt.tight_layout()
plt.savefig(FIGURES_DIR / "figure1_contact_rates.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 1 saved.")

### 4.2 RQ1 — Logistic regression: bot roles and newcomer retention

Three models are estimated following Zheng et al. (2019):

- **Model 1 (baseline):** session_edits + contact_human only
- **Model 2 (full):** session_edits + contact_human + all bot role dummies
- **Model 3 (null baseline):** majority class predictor — for robustness

Human editors serve as the reference category for contact effects.
Results are reported as odds ratios with 95% confidence intervals.

In [ ]:
def fit_logistic_model(
    df: pd.DataFrame,
    predictors: list[str],
    outcome: str = "retained",
) -> sm.regression.linear_model.RegressionResultsWrapper:
    """Fit a logistic regression model using statsmodels for interpretable output.

    Uses statsmodels rather than sklearn here because it provides coefficient
    tables, standard errors, p-values, and confidence intervals directly —
    essential for reporting in APA 7th edition format.

    Parameters
    ----------
    df         : Analysis dataframe.
    predictors : List of predictor column names.
    outcome    : Binary outcome column name.

    Returns
    -------
    Fitted statsmodels logit result object.
    """
    # Add constant (intercept) manually — statsmodels requires this.
    X = sm.add_constant(df[predictors])
    y = df[outcome]
    model = sm.Logit(y, X)
    return model.fit(disp=False)  # disp=False suppresses convergence output


def format_results_table(result, model_name: str) -> pd.DataFrame:
    """Extract coefficients, odds ratios, and CIs into a clean dataframe.

    Parameters
    ----------
    result     : Fitted statsmodels logit result.
    model_name : Label for this model (used as column prefix).

    Returns
    -------
    pd.DataFrame with OR, 95% CI, and significance stars.
    """
    coef  = result.params
    ci    = result.conf_int()
    pvals = result.pvalues

    # Convert log-odds to odds ratios for interpretability.
    table = pd.DataFrame({
        "OR":    np.exp(coef),
        "CI_lo": np.exp(ci[0]),
        "CI_hi": np.exp(ci[1]),
        "p":     pvals,
    })

    # Add significance stars following APA convention.
    def stars(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        return ""

    table["sig"] = table["p"].apply(stars)
    table.index.name = "predictor"
    return table


# ── Model 1: baseline (session_edits + contact_human) ────────────────────────
model1_predictors = ["session_edits", "contact_human"]
result_m1 = fit_logistic_model(df, model1_predictors)
table_m1  = format_results_table(result_m1, "Model 1")

print("=" * 60)
print("Model 1 — Baseline (human contact only)")
print("=" * 60)
print(f"  Log-likelihood : {result_m1.llf:.1f}")
print(f"  AIC            : {result_m1.aic:.1f}")
print(f"  N              : {int(result_m1.nobs):,}")
print()
print(table_m1.to_string(float_format="{:.3f}".format))

# ── Model 2: full (session_edits + all contact dummies) ───────────────────────
model2_predictors = ["session_edits"] + ALL_CONTACT_COLS
result_m2 = fit_logistic_model(df, model2_predictors)
table_m2  = format_results_table(result_m2, "Model 2")

print()
print("=" * 60)
print("Model 2 — Full model (all bot role dummies)")
print("=" * 60)
print(f"  Log-likelihood : {result_m2.llf:.1f}")
print(f"  AIC            : {result_m2.aic:.1f}")
print(f"  N              : {int(result_m2.nobs):,}")
print()
print(table_m2.to_string(float_format="{:.3f}".format))

In [ ]:
# ── Combined results table (APA 7th edition style) ───────────────────────────
def build_combined_table(
    results: list,
    model_names: list[str],
) -> pd.DataFrame:
    """Combine multiple model results into one APA-style table.

    Parameters
    ----------
    results     : List of fitted statsmodels logit results.
    model_names : Display names for each model.

    Returns
    -------
    pd.DataFrame suitable for display and export.
    """
    frames = []
    for result, name in zip(results, model_names):
        t = format_results_table(result, name)
        t.columns = pd.MultiIndex.from_tuples(
            [(name, c) for c in t.columns]
        )
        frames.append(t)
    return pd.concat(frames, axis=1)


combined = build_combined_table(
    [result_m1, result_m2],
    ["Model 1", "Model 2"]
)
print("Table 1. Logistic Regression Results: Bot Roles on Newcomer Retention (N = 5,845)")
print()
combined

In [ ]:
# ── Figure 2: Odds ratios from Model 2 ───────────────────────────────────────
# Exclude intercept and session_edits for cleaner visualisation of contact effects.
plot_vars = [p for p in model2_predictors if p != "session_edits"]
plot_data = table_m2.loc[plot_vars].copy()

clean_labels = {
    "contact_human":   "Human (reference group)",
    "contact_fixer":   "Fixer",
    "contact_tagger":  "Tagger",
    "contact_notifier":"Notifier",
    "contact_clerk":   "Clerk",
    "contact_advisor": "Advisor",
}
plot_data.index = [clean_labels.get(i, i) for i in plot_data.index]

fig, ax = plt.subplots(figsize=(7, 4))

# Plot odds ratios with 95% CI error bars.
for i, (label, row) in enumerate(plot_data.iterrows()):
    color = "#d62728" if row["p"] < 0.05 else "#aec7e8"
    ax.errorbar(
        row["OR"], i,
        xerr=[[row["OR"] - row["CI_lo"]], [row["CI_hi"] - row["OR"]]],
        fmt="o", color=color, capsize=4, markersize=6,
    )

ax.axvline(1, color="black", linewidth=0.8, linestyle="--",
           label="OR = 1 (no effect)")
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels(plot_data.index)
ax.set_xlabel("Odds Ratio (95% CI)")
ax.set_title(
    "Figure 2. Odds ratios for newcomer retention by sender type\n"
    "(Model 2; red = p < .05, blue = n.s.)"
)
ax.invert_yaxis()
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "figure2_odds_ratios.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 2 saved.")

### 4.3 RQ2 — Politeness profiles across sender types

Descriptively compare the 21 politeness strategy scores across sender types.
Following the dossier, this step is primarily descriptive and qualitative —
examining which strategies are most prevalent per sender type and what this
suggests about how different actors communicate with newcomers.

In [ ]:
def load_politeness_profiles(path: Path = POLITENESS_PROFILES_CSV) -> pd.DataFrame:
    """Load newcomer-level politeness profiles from notebook 03.

    Parameters
    ----------
    path : Path to politeness_profiles.csv.

    Returns
    -------
    pd.DataFrame with one row per newcomer and strategy score columns.

    Raises
    ------
    FileNotFoundError
        If politeness_profiles.csv does not exist. Run notebook 03 first.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Politeness profiles not found at {path}.\n"
            "Run notebook 03 first to generate them."
        )
    return pd.read_csv(path)


df_profiles = load_politeness_profiles()
print(f"Politeness profiles: {len(df_profiles):,} newcomers x {len(df_profiles.columns)} columns")
print(f"Sample columns: {list(df_profiles.columns[:6])}")

In [ ]:
def extract_sender_strategy_means(df_profiles: pd.DataFrame) -> pd.DataFrame:
    """Compute mean politeness strategy scores per sender type.

    Reshapes the wide-format profiles dataframe into a sender_type x strategy
    matrix for easy comparison and plotting.

    Parameters
    ----------
    df_profiles : Newcomer-level politeness profiles from notebook 03.

    Returns
    -------
    pd.DataFrame with sender types as rows and strategies as columns.
    """
    # Column names follow pattern: {sender_type}__{strategy}
    strategy_data = {}
    for col in df_profiles.columns:
        if "__" not in col:
            continue
        sender_type, strategy = col.split("__", 1)
        strategy_data.setdefault(sender_type, {})[strategy] = df_profiles[col].mean()

    return pd.DataFrame(strategy_data).T


df_strategy_means = extract_sender_strategy_means(df_profiles)
print("Mean politeness strategy scores by sender type:")
print(df_strategy_means.round(3).to_string())

In [ ]:
# ── Figure 3: Politeness heatmap ─────────────────────────────────────────────
# Show top 10 strategies by overall mean for readability.
top_strategies = df_strategy_means.mean().nlargest(10).index
df_plot = df_strategy_means[top_strategies]

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(df_plot.values, aspect="auto", cmap="YlOrRd")

ax.set_xticks(range(len(top_strategies)))
ax.set_xticklabels(
    [s.replace("feature_politeness_strategy_", "").replace("_", " ")
     for s in top_strategies],
    rotation=40, ha="right", fontsize=8
)
ax.set_yticks(range(len(df_plot.index)))
ax.set_yticklabels(df_plot.index)
ax.set_title(
    "Figure 3. Mean politeness strategy scores by sender type\n"
    "(top 10 strategies by overall mean; darker = higher score)"
)
plt.colorbar(im, ax=ax, label="Mean score")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "figure3_politeness_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 3 saved.")

In [ ]:
# ── Model 3: RQ2 extended model (contact dummies + politeness profiles) ────────
# Merge politeness profiles into analysis dataset for newcomers who received
# scored messages. NaN filled with 0 (newcomer received no messages from
# that sender type, or messages could not be scored).
df_extended = df.merge(df_profiles, on="user_id", how="left")
df_extended = df_extended.fillna(0)

# Use only strategy columns as additional predictors alongside contact dummies.
strategy_cols = [c for c in df_profiles.columns if "__" in c]
model3_predictors = ["session_edits"] + ALL_CONTACT_COLS + strategy_cols

result_m3 = fit_logistic_model(df_extended, model3_predictors)
table_m3  = format_results_table(result_m3, "Model 3")

print("=" * 60)
print("Model 3 — Extended (contact dummies + politeness profiles)")
print("=" * 60)
print(f"  Log-likelihood : {result_m3.llf:.1f}")
print(f"  AIC            : {result_m3.aic:.1f}")
print(f"  N              : {int(result_m3.nobs):,}")
print()
# Show only significant predictors for readability
sig_predictors = table_m3[table_m3["p"] < 0.05]
print(f"Significant predictors (p < .05): {len(sig_predictors)}")
print(sig_predictors.to_string(float_format="{:.3f}".format))

### 4.4 Robustness checks

Three robustness checks following the dossier:

1. **Null baseline** — majority class predictor to confirm bot-role contact
   adds explanatory power beyond class imbalance
2. **Cross-validation** — 5-fold stratified CV with precision, recall, macro-F1
3. **Stricter retention threshold** — re-operationalise retention as ≥3 edits
   instead of ≥1 to test construct validity

In [ ]:
def evaluate_model_cv(
    df: pd.DataFrame,
    predictors: list[str],
    outcome: str = "retained",
    n_folds: int = N_CV_FOLDS,
    seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    """Evaluate a logistic regression model with stratified k-fold CV.

    Reports accuracy, precision, recall, and macro-F1 to account for
    class imbalance (86.3% non-retained vs 13.7% retained).

    Parameters
    ----------
    df         : Analysis dataframe.
    predictors : List of predictor column names.
    outcome    : Binary outcome column name.
    n_folds    : Number of CV folds.
    seed       : Random seed for fold assignment.

    Returns
    -------
    pd.DataFrame with mean and SD for each metric across folds.
    """
    X = df[predictors].values
    y = df[outcome].values

    # Standardise predictors — required for logistic regression convergence
    # when predictors are on different scales (session_edits vs binary dummies).
    scaler = StandardScaler()
    X      = scaler.fit_transform(X)

    clf = LogisticRegression(random_state=seed, max_iter=1000)
    cv  = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    scoring = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
    cv_results = cross_validate(clf, X, y, cv=cv, scoring=scoring)

    summary = pd.DataFrame({
        metric: {
            "mean": cv_results[f"test_{metric}"].mean(),
            "sd":   cv_results[f"test_{metric}"].std(),
        }
        for metric in scoring
    }).T

    return summary


# ── Null baseline ─────────────────────────────────────────────────────────────
majority_class_accuracy = max(df["retained"].mean(), 1 - df["retained"].mean())
print(f"Null baseline accuracy (majority class): {majority_class_accuracy:.3f}")

# ── Cross-validation: Model 1 vs Model 2 ─────────────────────────────────────
print()
print("5-fold stratified CV — Model 1 (baseline):")
cv_m1 = evaluate_model_cv(df, model1_predictors)
print(cv_m1.round(3).to_string())

print()
print("5-fold stratified CV — Model 2 (full):")
cv_m2 = evaluate_model_cv(df, model2_predictors)
print(cv_m2.round(3).to_string())

# ── Stricter retention threshold ──────────────────────────────────────────────
print()
print("Robustness check: stricter retention (≥3 edits)")
# Note: this requires re-fetching edit counts post-window — approximate here
# using total_edits as a proxy if available, or flag as requiring re-collection.
if "total_edits" in df.columns:
    df["retained_strict"] = (df["total_edits"] >= 3).astype(int)
    result_strict = fit_logistic_model(df, model2_predictors, outcome="retained_strict")
    table_strict  = format_results_table(result_strict, "Strict")
    print(f"Strict retention rate: {df['retained_strict'].mean():.1%}")
    print(table_strict.to_string(float_format="{:.3f}".format))
else:
    print("  total_edits not available in analysis dataset — strict threshold")
    print("  requires re-fetching post-window edit counts. Skipped.")

### 4.5 Summary

In [ ]:
print("=" * 60)
print("Analysis complete — key results")
print("=" * 60)
print(f"  N newcomers          : {len(df):,}")
print(f"  Retention rate       : {df['retained'].mean():.1%}")
print()
print("Model comparison:")
print(f"  Model 1 AIC          : {result_m1.aic:.1f}")
print(f"  Model 2 AIC          : {result_m2.aic:.1f}")
print(f"  Model 3 AIC          : {result_m3.aic:.1f}")
print()
print("Figures saved:")
for fig_path in sorted(FIGURES_DIR.glob("*.png")):
    print(f"  {fig_path.name}")